# Setup

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, 
    accuracy_score, 
    confusion_matrix,
    classification_report
)
from simpletransformers.classification import ClassificationModel, ClassificationArgs
#import wandb
#import numpy as np
#from scipy.special import softmax

Configure project paths. Adjust based on notebook location.

In [2]:
project_root = Path.cwd().parent if "Scripts" in Path.cwd().parts else Path.cwd()
print(f"Project root: {project_root}")

data_dir = project_root / 'Data'
output_dir = project_root / 'Model-Outputs'

data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")

Project root: C:\Users\sile9\Documents\projects\parldebates_analysis
Data directory: C:\Users\sile9\Documents\projects\parldebates_analysis\Data
Output directory: C:\Users\sile9\Documents\projects\parldebates_analysis\Model-Outputs


# Import Data and Preprocessing

In [3]:
dat = pd.read_csv(data_dir / '04_labeled_dataset_cleaned.csv')

# Ensure target variable is binary (0, 1)
dat['climate'] = dat['climate'].astype(int)
dat["label"] = dat["climate"].astype("category").cat.codes

# Prepare dataframe
dat_prepared = dat[['paragraph_text', 'label']].copy()
dat_prepared.columns = ['text', 'labels']

# Create train/test/validation splits
train_df, test_df = train_test_split(
    dat_prepared, 
    test_size=0.25, 
    stratify=dat_prepared['labels'], 
    random_state=42
)
val_df, test_df = train_test_split(
    test_df, 
    test_size=0.5, 
    stratify=test_df['labels'], 
    random_state=42
)

# Check Class Distribution

In [4]:
print("\n" + "="*60)
print("CLASS DISTRIBUTION")
print("="*60)
print("Training set:")
print(train_df['labels'].value_counts())
print(f"\nClass 0: {(train_df['labels']==0).sum()} samples")
print(f"Class 1: {(train_df['labels']==1).sum()} samples")
print(f"Imbalance ratio: {(train_df['labels']==0).sum() / (train_df['labels']==1).sum():.2f}:1")
print("="*60)


CLASS DISTRIBUTION
Training set:
labels
0    293
1     82
Name: count, dtype: int64

Class 0: 293 samples
Class 1: 82 samples
Imbalance ratio: 3.57:1


# Compute Class Weights

In [5]:
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['labels']),
    y=train_df['labels']
)

weights_list = class_weights.tolist()

print("\n" + "="*60)
print("CLASS WEIGHTS (to handle imbalance)")
print("="*60)
print(f"Class 0 weight: {weights_list[0]:.4f}")
print(f"Class 1 weight: {weights_list[1]:.4f}")
print("="*60)


CLASS WEIGHTS (to handle imbalance)
Class 0 weight: 0.6399
Class 1 weight: 2.2866


# Model Configuration

In [6]:
model_type = "xlmroberta"
model_name = "xlm-roberta-base"

model_args = ClassificationArgs()
model_args.num_train_epochs = 5  # Increased
model_args.train_batch_size = 8
model_args.eval_batch_size = 8
model_args.max_seq_length = 256
model_args.learning_rate = 3e-5  # Slightly higher
model_args.overwrite_output_dir = True
model_args.evaluate_during_training = True
model_args.evaluate_during_training_steps = 100  # Monitor frequently
model_args.evaluate_during_training_verbose = True
model_args.use_multiprocessing = False
model_args.save_eval_checkpoints = False
model_args.save_model_every_epoch = False
model_args.manual_seed = 42
model_args.weight = weights_list  # ← ADD CLASS WEIGHTS

# Point model outputs to output_dir
model_args.output_dir = str(output_dir / 'model')
model_args.best_model_dir = str(output_dir / 'model' / 'best_model')
model_args.cache_dir = str(output_dir / 'model' / 'cache')

# Early stopping
model_args.use_early_stopping = True
model_args.early_stopping_patience = 3
model_args.early_stopping_metric = "eval_loss"

# Training Function

In [7]:
def train():
    """Training function for binary classification model"""
    
    #wandb.init(project="binary_classification_fixed")
    
    model = ClassificationModel(
        model_type,
        model_name,
        num_labels=2,
        use_cuda=False,
        args=model_args,
    )
    
    print("\n" + "="*60)
    print("🚀 Starting training with CLASS WEIGHTS...")
    print("="*60)
    
    model.train_model(train_df, eval_data=val_df)
    
    print("\n" + "="*60)
    print("📊 Evaluating on validation set...")
    print("="*60)
    
    # Validation evaluation
    val_predictions, _ = model.predict(val_df['text'].tolist())
    val_labels = val_df['labels'].values
    
    val_report = classification_report(val_labels, val_predictions,
                                       target_names=['Class 0', 'Class 1'])
    val_cm = confusion_matrix(val_labels, val_predictions)
    
    print("\nVALIDATION RESULTS:")
    print("="*60)
    print(val_report)
    print("\nConfusion Matrix:")
    print(val_cm)
    
    # Test evaluation
    print("\n" + "="*60)
    print("📊 Evaluating on test set...")
    print("="*60)
    
    test_predictions, _ = model.predict(test_df['text'].tolist())
    test_labels = test_df['labels'].values
    
    test_report = classification_report(test_labels, test_predictions,
                                        target_names=['Class 0', 'Class 1'])
    test_cm = confusion_matrix(test_labels, test_predictions)
    test_accuracy = accuracy_score(test_labels, test_predictions)
    test_f1 = f1_score(test_labels, test_predictions)
    
    print("\nTEST RESULTS:")
    print("="*60)
    print(test_report)
    print("\nConfusion Matrix:")
    print(test_cm)
    print("="*60)
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test F1 Score: {test_f1:.4f}")
    print("="*60)

    # Save results to output_dir 
    results_path = output_dir / 'evaluation_results.txt'
    with open(results_path, 'w') as f:
        f.write("VALIDATION RESULTS\n")
        f.write("="*60 + "\n")
        f.write(val_report)
        f.write("\nConfusion Matrix:\n")
        f.write(str(val_cm))
        f.write("\n\n")
        f.write("TEST RESULTS\n")
        f.write("="*60 + "\n")
        f.write(test_report)
        f.write("\nConfusion Matrix:\n")
        f.write(str(test_cm))
        f.write(f"\n\nTest Accuracy: {test_accuracy:.4f}\n")
        f.write(f"Test F1 Score: {test_f1:.4f}\n")
    print(f"\n💾 Evaluation results saved to {results_path}")

    # Save predictions as CSV
    pd.DataFrame({
        'text': test_df['text'].values,
        'true_label': test_labels,
        'predicted_label': test_predictions
    }).to_csv(output_dir / 'test_predictions.csv', index=False)
    print(f"💾 Test predictions saved to {output_dir / 'test_predictions.csv'}")

    #wandb.finish()
    return model

In [8]:
#wandb.finish()

# Run Training

In [9]:
if __name__ == "__main__":
    best_model_path = output_dir / 'model' / 'best_model'
    
    if best_model_path.exists():
        print("✅ Trained model found, loading instead of retraining...")
        trained_model = ClassificationModel(
            "xlmroberta",
            str(best_model_path),
            num_labels=2,
            use_cuda=False,
        )
    else:
        print("🚀 No trained model found, starting training...")
        trained_model = train()
        print("\n✅ Training completed!")

🚀 No trained model found, starting training...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Starting training with CLASS WEIGHTS...


C:\Users\sile9\anaconda3\envs\model-training\Lib\site-packages\simpletransformers\classification\classification_model.py:490: UserWarning: use_multiprocessing automatically disabled as xlmroberta fails when using multiprocessing for feature conversion.
  warnings.warn(


Map:   0%|          | 0/375 [00:00<?, ? examples/s]

...

Predicting:   0%|          | 0/8 [00:00<?, ?it/s]


VALIDATION RESULTS:
              precision    recall  f1-score   support

     Class 0       0.96      0.94      0.95        49
     Class 1       0.79      0.85      0.81        13

    accuracy                           0.92        62
   macro avg       0.87      0.89      0.88        62
weighted avg       0.92      0.92      0.92        62


Confusion Matrix:
[[46  3]
 [ 2 11]]

📊 Evaluating on test set...


0it [00:00, ?it/s]

Predicting:   0%|          | 0/8 [00:00<?, ?it/s]


TEST RESULTS:
              precision    recall  f1-score   support

     Class 0       0.98      0.96      0.97        49
     Class 1       0.87      0.93      0.90        14

    accuracy                           0.95        63
   macro avg       0.92      0.94      0.93        63
weighted avg       0.95      0.95      0.95        63


Confusion Matrix:
[[47  2]
 [ 1 13]]
Test Accuracy: 0.9524
Test F1 Score: 0.8966

💾 Evaluation results saved to C:\Users\sile9\Documents\projects\parldebates_analysis\Model-Outputs\evaluation_results.txt
💾 Test predictions saved to C:\Users\sile9\Documents\projects\parldebates_analysis\Model-Outputs\test_predictions.csv

✅ Training completed!


# Import Data for Classification

In [12]:
dat_class = pd.read_csv(data_dir / '02_paragraphs_cleaned.csv')

In [13]:
dat_class_prepared = dat_class[['paragraph_id', 'paragraph_text']].copy()
dat_class_prepared.columns = ['id', 'text']
dat_class_prepared.head()

,id,text
0,191151-1,Vous avez reçu le rapport du Conseil fédéral s...
1,191151-2,En ce qui concerne le rapport du Conseil fédér...
2,191151-3,S'agissant de la constitution du Conseil natio...
3,191152-1,"Präsident (Stamm Luzi, Alterspräsident): Dem A..."
4,191152-2,Gemäss Artikel 4 unseres Ratsreglementes hat d...


In [15]:
dat_class_prepared.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303931 entries, 0 to 303930
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      303931 non-null  object
 1   text    303928 non-null  object
dtypes: object(2)
memory usage: 4.6+ MB


In [16]:
dat_class_prepared['token_length'] = dat_class_prepared['text'].str.split().str.len()
print(dat_class_prepared['token_length'].describe())

count    303928.000000
mean         72.328713
std          42.028553
min           0.000000
25%          41.000000
50%          67.000000
75%          98.000000
max         623.000000
Name: token_length, dtype: float64


In [17]:
over_limit = (dat_class_prepared['token_length'] > 256).mean() * 100
print(f"{over_limit:.1f}% of texts exceed 256 tokens and will be truncated")

0.1% of texts exceed 256 tokens and will be truncated


In [24]:
print(dat_class_prepared['text'].isna().sum())  # check how many

3


In [25]:
dat_class_prepared = dat_class_prepared.dropna(subset=['text'])

# Run Predictions

In [22]:
#increase batch size
trained_model.args.eval_batch_size = 64
trained_model.args.use_multiprocessing = False
trained_model.args.use_multiprocessing_for_evaluation = False

In [26]:
predictions, raw_outputs = trained_model.predict(
    dat_class_prepared['text'].tolist()
)

dat_class_prepared['predicted_label'] = predictions

dat_class_prepared.to_csv(output_dir / 'classified_paragraphs.csv', index=False)
print(f"✅ Classified {len(dat_class_prepared)} paragraphs")

Map:   0%|          | 0/303928 [00:00<?, ? examples/s]

Predicting:   0%|          | 0/4749 [00:00<?, ?it/s]

✅ Classified 303928 paragraphs
